# Compare Different Models on Real-World Datasets

Evaluate all registered models on OpenML benchmark datasets, with per-model cache reuse and W&B logging.

In [ ]:
from pathlib import Path

from pfns.run_evaluation_cli import (
    build_available_baseline_model_configs,
    compute_per_dataset_stats,
    print_results_summary,
    run_real_world_model_from_config,
    summarize_results,
)
from pfns.experiments.model_benchmarks.hashing import (
    single_model_hash,
)
from pfns.utils import get_default_device
from pfns.experiments.model_benchmarks.io import (
    REAL_WORLD_BUNDLE_KEYS,
    REAL_WORLD_REQUIRED_FILES,
    download_results_bundle_from_wandb,
    find_latest_real_world_bundle_for_model,
    load_dataframe_bundle,
    make_bundle_path,
    make_model_artifact_name,
    sanitize_wandb_artifact_component,
    save_dataframe_bundle,
    upload_results_bundle_to_wandb,
)
from pfns.experiments.model_benchmarks.model_registry import (
    MODEL_FAMILIES,
    get_all_models,
    get_baseline_models,
    get_models_from_families,
    get_models_from_names,
)
from pfns.experiments.model_benchmarks.workflows import (
    aggregate_real_world_results_from_bundles,
    build_real_world_run_metadata,
    real_world_bundle_is_compatible,
)

REAL_DATASET_EXPERIMENT = {
    "name": "real_world_openml_comparison",
    "benchmark": "opencc",
    "max_samples": 1000,
    "max_features": 20,
    "max_classes": 10,
    "n_splits": 5,
    "batch_size_inference": 32,
    "n_ensemble_configurations": 32,
    "preprocess_transforms": ["none", "power", "robust"],
    "sample_order_permutation": True,
    "fla_cache_chunk_size": None,
}

BASELINE_EVAL = {
    "n_jobs": 4,
    "random_state": 42,
}

WANDB = {
    "enabled": True,
    "overwrite": False,
    "artifact_name_real_eval": "real_eval_results",
    "entity": "icl_arch",
    "artifact_project": "real_world_eval_artifacts",
    "scores_project": "real_world_eval_scores",
    "mode": "online"
}

OUTPUT_ROOT = Path.cwd().resolve() / "exp_outputs" / "real_world_eval"
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

print(f"Results are stored in: {OUTPUT_ROOT}")
print(f"Available model families: {list(MODEL_FAMILIES)}")

# Model selection
# models_to_compare = get_models_from_families(["transformer", "kda"])
# models_to_compare = get_models_from_names(["Softmax_Transformer", "GLA_Cached"])
checkpoint_models_to_compare = get_all_models()
baseline_models_to_compare, skipped_baselines = build_available_baseline_model_configs(
    candidates=get_baseline_models(),
    n_jobs=BASELINE_EVAL["n_jobs"],
    random_state=BASELINE_EVAL["random_state"],
)
if skipped_baselines:
    print(f"Skipping unavailable baselines in this environment: {skipped_baselines}")

models_to_compare = {
    **checkpoint_models_to_compare,
    **baseline_models_to_compare,
}

print(
    "Selected models:",
    len(models_to_compare),
    f"({len(checkpoint_models_to_compare)} checkpoints + {len(baseline_models_to_compare)} baselines)",
)

device = get_default_device()
print(f"Using device: {device}")

expected_real_metadata = build_real_world_run_metadata(
    experiment=REAL_DATASET_EXPERIMENT,
    device=device,
)

In [ ]:
if WANDB["enabled"] and WANDB["overwrite"]:
    print("WANDB overwrite=True: skipping per-model download and forcing reruns.")

for model_name, model_config in models_to_compare.items():
    model_hash = single_model_hash(
        model_name=model_name,
        model_config=model_config,
        experiment_payload=REAL_DATASET_EXPERIMENT,
    )
    model_artifact_name = make_model_artifact_name(
        base_artifact_name=WANDB["artifact_name_real_eval"],
        model_name=model_name,
        model_hash=model_hash,
    )

    if WANDB["enabled"] and not WANDB["overwrite"]:
        cached_bundle_path = download_results_bundle_from_wandb(
            artifact_name=model_artifact_name,
            download_root=OUTPUT_ROOT / "wandb_model_cache" / "real_world",
            required_files=REAL_WORLD_REQUIRED_FILES,
            entity=WANDB["entity"],
            project=WANDB["artifact_project"],
        )
        print(f"Checked for cached W&B artifact for {model_name}: {cached_bundle_path}")
        if cached_bundle_path is not None:
            cached_bundle = load_dataframe_bundle(
                cached_bundle_path,
                expected_keys=REAL_WORLD_BUNDLE_KEYS,
            )
            if real_world_bundle_is_compatible(
                cached_bundle,
                model_name=model_name,
                expected_metadata=expected_real_metadata,
            ):
                print(f"Reused cached real-world W&B result for {model_name}: {cached_bundle_path}")
                continue

    print(f"Running real-world benchmark for model: {model_name}")
    results = run_real_world_model_from_config(
        model_config=model_config,
        experiment=REAL_DATASET_EXPERIMENT,
        device=device,
        baseline_n_jobs=BASELINE_EVAL["n_jobs"],
        random_state=BASELINE_EVAL["random_state"],
        verbose=False,
    )

    if not results.empty:
        results["model"] = model_name
    else:
        print(f"Warning: No results for model {model_name}, skipping saving and upload.")
        continue
    summary = summarize_results(results)
    per_dataset = compute_per_dataset_stats(results)

    model_bundle_path = make_bundle_path(
        OUTPUT_ROOT / "real_world",
        f"{REAL_DATASET_EXPERIMENT['name']}_{sanitize_wandb_artifact_component(model_name)}",
    )
    save_dataframe_bundle(
        dataframes={
            "results": results,
            "summary": summary.reset_index() if summary is not None else None,
            "per_dataset": per_dataset,
        },
        bundle_dir=model_bundle_path,
        experiment=REAL_DATASET_EXPERIMENT,
        run_metadata=expected_real_metadata,
    )
    print(f"Saved real-world bundle for {model_name}: {model_bundle_path}")

    if WANDB["enabled"]:
        artifact_ref = upload_results_bundle_to_wandb(
            model_bundle_path,
            artifact_name=model_artifact_name,
            run_name=f"{REAL_DATASET_EXPERIMENT['name']}_{sanitize_wandb_artifact_component(model_name)}_{model_hash}_artifact",
            metadata={
                "experiment": REAL_DATASET_EXPERIMENT,
                "model_name": model_name,
                "model_config": model_config,
                "model_hash": model_hash,
                "run_metadata": expected_real_metadata,
            },
            entity=WANDB["entity"],
            project=WANDB["artifact_project"],
            job_type="real_world_bundle_upload",
        )
        print(f"Uploaded real-world artifact for {model_name}: {artifact_ref}")

print("\nReal-world evaluation completed.")



## Load and aggregate per-model outputs

Collect the latest compatible bundle for each selected model, then build combined result tables.

In [ ]:
from IPython.display import display

from pfns.datasets.tabular_datasets import (
    open_cc_dids as OPENCC_BENCHMARK,
    test_dids_classification as TEST_BENCHMARK,
)


bundles_by_model = {}
missing_models = []

for model_name in models_to_compare:
    bundle_path, bundle = find_latest_real_world_bundle_for_model(
        model_name,
        output_root=OUTPUT_ROOT,
        expected_metadata=expected_real_metadata,
    )
    if bundle is None:
        missing_models.append(model_name)
        continue

    bundles_by_model[model_name] = {"path": bundle_path, "bundle": bundle}
    print(f"Loaded {model_name} bundle: {bundle_path}")

if not bundles_by_model:
    raise RuntimeError(
        "No compatible bundles were found. Run the benchmark cell first, or check metadata settings."
    )

all_results, results_by_model = aggregate_real_world_results_from_bundles(
    bundles_by_model,
    expected_splits=int(REAL_DATASET_EXPERIMENT["n_splits"]),
)

summary = summarize_results(all_results)
per_dataset = compute_per_dataset_stats(all_results)
if summary is None or per_dataset is None or per_dataset.empty:
    raise RuntimeError("Could not compute summary tables from aggregated results.")

benchmark_ids = (
    OPENCC_BENCHMARK
    if REAL_DATASET_EXPERIMENT["benchmark"] == "opencc"
    else TEST_BENCHMARK
)

print("\nAggregated real-world benchmark summary")
print(f"Models loaded: {len(bundles_by_model)} / {len(models_to_compare)}")
print(f"Rows in all_results: {len(all_results)}")
print(f"Datasets covered: {per_dataset['dataset'].nunique()} / {len(benchmark_ids)}")

if missing_models:
    print("Missing compatible bundles for:", missing_models)

print_results_summary(all_results, title="Aggregated Results Across Selected Models")


## Result tables

Display model-level and dataset-level metric tables from the aggregated benchmark results.

In [ ]:
if summary is None or per_dataset is None or summary.empty or per_dataset.empty:
    raise RuntimeError("No summary/per-dataset results available.")

summary_display = summary.copy().sort_values("accuracy_mean", ascending=False)
print("Model summary table (mean over datasets):")
display(
    summary_display.style.format(
        {
            "accuracy_mean": "{:.4f}",
            "accuracy_std": "{:.4f}",
            "roc_auc_mean": "{:.4f}",
            "roc_auc_std": "{:.4f}",
            "log_loss_mean": "{:.4f}",
            "log_loss_std": "{:.4f}",
            "ece_mean": "{:.4f}",
            "ece_std": "{:.4f}",
            "fit_time_mean": "{:.3f}",
            "predict_time_mean": "{:.3f}",
        }
    ).background_gradient(cmap="Greens_r", subset=["accuracy_mean", "roc_auc_mean"])
)

for metric in ["accuracy_mean", "roc_auc_mean", "log_loss_mean", "ece_mean"]:
    print(f"\nPer-dataset table: {metric}")
    table = (
        per_dataset
        .pivot_table(index="dataset", columns="model", values=metric, observed=True)
        .sort_index()
    )
    display(table.style.format("{:.4f}").background_gradient(cmap="Blues"))


## Plots

Create metric bars, speed-vs-accuracy tradeoff, and dataset-level rank curves using existing benchmark utilities.

In [ ]:
import matplotlib.pyplot as plt

from pfns.experiments.model_benchmarks.analysis import compute_mean_rank_tables
from pfns.experiments.model_benchmarks.plotting import build_model_style_map, plot_curves_from_df

if summary is None or summary.empty:
    raise RuntimeError("No summary dataframe available for plotting.")

ordered_models = summary.sort_values("accuracy_mean", ascending=False).index.tolist()
style_map = build_model_style_map(ordered_models)
model_colors = {name: style_map[name][2] for name in ordered_models}

# 1) Summary metric bar plots
bar_specs = [
    ("accuracy_mean", "accuracy_std", "Accuracy", True),
    ("roc_auc_mean", "roc_auc_std", "ROC-AUC", True),
    ("log_loss_mean", "log_loss_std", "LogLoss", False),
    ("ece_mean", "ece_std", "ECE", False),
]

fig, axes = plt.subplots(2, 2, figsize=(16, 10), dpi=130)
for ax, (mean_col, std_col, title, higher_is_better) in zip(axes.flat, bar_specs):
    plot_df = summary.reset_index().rename(columns={"index": "model"})
    plot_df = plot_df.sort_values(mean_col, ascending=not higher_is_better)

    ax.barh(
        plot_df["model"],
        plot_df[mean_col],
        xerr=plot_df[std_col],
        color=[model_colors[m] for m in plot_df["model"]],
        alpha=0.9,
    )
    ax.set_title(title)
    ax.grid(axis="x", alpha=0.25)

plt.tight_layout()
plt.show()

# 2) Accuracy vs inference speed tradeoff
tradeoff_df = summary.reset_index().rename(columns={"index": "model"})
fig, ax = plt.subplots(figsize=(9, 6), dpi=130)
for _, row in tradeoff_df.iterrows():
    model_name = row["model"]
    ax.scatter(
        row["predict_time_mean"],
        row["accuracy_mean"],
        s=90,
        color=model_colors[model_name],
        alpha=0.9,
        label=model_name,
    )
    ax.annotate(model_name, (row["predict_time_mean"], row["accuracy_mean"]), xytext=(4, 4), textcoords="offset points", fontsize=9)

ax.set_xlabel("Predict time mean (s)")
ax.set_ylabel("Accuracy mean")
ax.set_xscale("log")
ax.grid(alpha=0.25)
ax.set_title("Accuracy vs Prediction Time")
plt.show()

# 3) Dataset-wise rank curves with model_benchmarks rank + plotting helpers
rank_base = (
    all_results
    .groupby(["model", "dataset"], observed=True)
    .agg(
        accuracy=("accuracy", "mean"),
        roc_auc=("roc_auc", "mean"),
        log_loss=("log_loss", "mean"),
        ece=("ece", "mean"),
    )
    .reset_index()
)

rank_long = rank_base.melt(
    id_vars=["model", "dataset"],
    value_vars=["accuracy", "roc_auc", "log_loss", "ece"],
    var_name="metric",
    value_name="value",
)
rank_long["metric"] = rank_long["metric"].replace({"accuracy": "acc"})

dataset_to_idx = {
    name: idx
    for idx, name in enumerate(sorted(rank_long["dataset"].unique()), start=1)
}
rank_input = rank_long.assign(seqlen=rank_long["dataset"].map(dataset_to_idx))
rank_input["rep"] = rank_input["dataset"]

rank_tables = compute_mean_rank_tables(
    rank_input,
    x_col="seqlen",
    higher_is_better_metrics={"acc", "roc_auc"},
)

overall_ranks = rank_tables["overall_ranks"].copy()
overall_ranks["metric"] = overall_ranks["metric"].replace({"acc": "accuracy"})
print("Overall mean rank (lower is better):")
for metric in ["accuracy", "roc_auc", "log_loss", "ece"]:
    ranked = (
        overall_ranks[overall_ranks["metric"] == metric]
        .sort_values("rank")
        .drop(columns="metric")
        .reset_index(drop=True)
    )
    print(f"\n{metric}")
    display(ranked.style.format({"rank": "{:.2f}"}).background_gradient(subset=["rank"], cmap="Greens_r"))

rank_curve_df = rank_tables["x_ranks"].copy()
rank_curve_df["metric"] = rank_curve_df["metric"].replace(
    {
        "acc": "accuracy_rank",
        "roc_auc": "roc_auc_rank",
        "log_loss": "log_loss_rank",
        "ece": "ece_rank",
    }
)

plot_curves_from_df(
    rank_curve_df,
    specs=[
        ("accuracy_rank", "Accuracy Rank"),
        ("roc_auc_rank", "ROC-AUC Rank"),
        ("log_loss_rank", "LogLoss Rank"),
        ("ece_rank", "ECE Rank"),
    ],
    style_map=style_map,
    x_col="seqlen",
    value_col="rank",
    x_label="Dataset index (sorted by dataset name)",
    title_suffix=" (1 is best)",
    invert_y=True,
    show_std=False,
    figsize=(24, 7),
)
